<a href="https://colab.research.google.com/github/prakashb-ai/Cal-Ai-/blob/main/Data_Analayst_Intern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Rasoi Growth Analytics — Data Analyst Intern Take-Home
Working notebook / script. Recomputes all metrics from rasoi_growth_data.csv.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from pathlib import Path

DATA_PATH = Path('/content/rasoi_growth_data.csv')
OUT_DIR = Path('/content')

# ---------------------------------------------------------------------------
# 1. Load & profile
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
assert df.shape == (540, 12), f"Unexpected shape {df.shape}"
assert df.isnull().sum().sum() == 0, "Unexpected missing values"

print("=== TASK 1: DATA PROFILE ===")
print(f"Rows: {len(df)}, Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Campaigns: {sorted(df['campaign'].unique())}")
print(f"Channels: {sorted(df['channel'].unique())}")
print("No missing values. Revenue exactly matches trials*1 + monthly*199 + annual*1199.")

# The one real problem
df['is_anomaly'] = (
    (df['channel'] == 'google')
    & (df['date'] >= '2026-07-19')
    & (df['date'] <= '2026-07-24')
)
anom = df[df['is_anomaly']]
print(f"\n*** ANOMALY FOUND: {len(anom)} rows ***")
print("Google campaigns (gg_uac_installs + gg_uac_actions) 19–24 Jul 2026:")
print("  - Spend, impressions, clicks, installs NORMAL")
print("  - Trials, renewals, revenue collapsed to ~1–2% of normal levels")
print("  - Trial CVR fell from ~6% to ~0.3–0.6%")
print("Likely cause: purchase-event tracking / attribution outage (or payment")
print("gateway issue) specific to Google traffic for those 6 days. Installs")
print("continued to be recorded (different MMP path), but in-app trial purchases")
print("were not attributed.")
print("Rows touched: all Google rows for dates 2026-07-19 through 2026-07-24.")
print("Handling: EXCLUDE these 12 rows from all conversion / revenue / ratio")
print("calculations (CPT, CAC, trial-to-paid, ROAS). Keep them in total spend")
print("and install counts because the money was spent and users installed.")
print("Alternative considered: impute trials from historical install→trial rate;")
print("chose exclusion for transparency and to avoid inventing conversions.")

# ---------------------------------------------------------------------------
# 2. Campaign metrics (full period, lag-aware where it matters)
# ---------------------------------------------------------------------------
print("\n=== TASK 2: CAMPAIGN METRICS ===")

def compute_metrics(df):
    rows = []
    for camp, g in df.groupby('campaign'):
        g = g.sort_values('date')
        clean = g[~g['is_anomaly']]
        spend = g['spend_inr'].sum()          # money spent
        installs = g['installs'].sum()
        trials = clean['trials'].sum()
        renewals = clean['renewals'].sum()
        annual = clean['annual_renewals'].sum()
        revenue = clean['revenue_inr'].sum()
        refunds = clean['refunds_inr'].sum()
        net_rev = revenue - refunds

        # Lag-aware trial→paid: renewals[d] belong to trials[d-1]
        g2 = clean.copy()
        g2['trials_prev'] = g2['trials'].shift(1)
        g2 = g2.dropna(subset=['trials_prev'])
        lag_t2p = g2['renewals'].sum() / g2['trials_prev'].sum() if g2['trials_prev'].sum() > 0 else np.nan

        rows.append({
            'campaign': camp,
            'channel': g['channel'].iloc[0],
            'spend_inr': spend,
            'installs': installs,
            'trials': trials,
            'renewals': renewals,
            'annual_renewals': annual,
            'net_revenue': net_rev,
            'cpi': spend / installs,
            'cpt': spend / trials,
            'cac': spend / renewals,
            'trial_to_paid': lag_t2p,
            'trial_to_paid_sum': renewals / trials,
            'annual_mix': annual / renewals,
            'first_month_roas': net_rev / spend,
            'net_rev_per_trial': net_rev / trials,
            'net_rev_per_payer': net_rev / renewals,
        })
    return pd.DataFrame(rows).set_index('campaign')

m = compute_metrics(df)
print(m[['channel','cpi','cpt','cac','trial_to_paid','annual_mix','first_month_roas']].round(3).to_string())
print("\nNote on the trap: same-day renewals/trials understates conversion when")
print("spend (and trials) is rising, because today's renewals come from")
print("yesterday's smaller trial cohort. Over 90 days the bias is small;")
print("we still use lag-matched rates for precision.")

# ---------------------------------------------------------------------------
# 3. Ranking & recommendation metric
# ---------------------------------------------------------------------------
print("\n=== TASK 3: RANKING ===")
print("By cost-per-trial (lowest first):")
print(m['cpt'].sort_values().round(1).to_string())
print("\nBy first-month net ROAS (highest first) — the metric I believe:")
print(m['first_month_roas'].sort_values(ascending=False).round(3).to_string())
print("\nOrder changes completely. mt_lookalike_3pct is cheapest on CPT but")
print("near-worst on ROAS because only 14% of trials become paying users and")
print("almost none take the annual plan. It is a volume trap: looks efficient")
print("on the top of funnel, expensive on the bottom.")
print("\nSteering metric I nominate: first-month net ROAS (net revenue / spend).")
print("Why: (1) it is the actual cash returned in the first billing cycle, the")
print("horizon that decides whether to keep scaling a campaign this month;")
print("(2) it automatically incorporates trial→paid conversion, plan mix, and")
print("refunds without needing a separate LTV model; (3) it is observable")
print("weekly with only a 1-day lag and does not require cohort tracking")
print("beyond what the current data already contains.")

# ---------------------------------------------------------------------------
# 4. Budget split for ₹1 Cr
# ---------------------------------------------------------------------------
print("\n=== TASK 4: BUDGET ALLOCATION (₹1 Cr) ===")

budget = 10_000_000
hist_monthly = {c: m.loc[c, 'spend_inr'] / 90 * 30 for c in m.index}

prop = {
    'mt_broad_aoa': 0.28,
    'gg_uac_actions': 0.25,
    'gg_uac_installs': 0.18,
    'mt_retarget_paywall': 0.12,
    'mt_lookalike_3pct': 0.12,
    'sc_feed_video': 0.05,
}

def dim_factor(scale):
    if scale <= 1.0:
        return 1.00
    if scale <= 1.5:
        return 0.95
    if scale <= 2.5:
        return 0.85
    return 0.70

plan_rows = []
for camp, share in prop.items():
    spend = budget * share
    scale = spend / hist_monthly[camp]
    dfact = dim_factor(scale)
    cpt = m.loc[camp, 'cpt']
    t2p = m.loc[camp, 'trial_to_paid']
    nrpp = m.loc[camp, 'net_rev_per_payer']
    trials = (spend / cpt) * dfact
    payers = trials * t2p
    net_rev = payers * nrpp
    plan_rows.append({
        'campaign': camp,
        'proposed_spend': int(round(spend)),
        'expected_trials': int(round(trials)),
        'expected_payers': int(round(payers)),
        'expected_net_rev': int(round(net_rev)),
        'implied_roas': round(net_rev / spend, 2),
        'scale_vs_hist': round(scale, 2),
        'dim_factor': dfact,
    })
plan = pd.DataFrame(plan_rows)
print(plan.to_string(index=False))
print(f"\nTotal expected payers: {plan['expected_payers'].sum():,}")
print(f"Total expected first-month net revenue: ₹{plan['expected_net_rev'].sum():,}")
print(f"Blended first-month ROAS: {plan['expected_net_rev'].sum() / budget:.2f}")

print("\nAssumptions:")
print("- Historical CPT, lag-aware trial→paid, and net-rev-per-payer remain")
print("  stable at the new spend levels (after applying dim_factor).")
print("- Diminishing returns: mild (5–15% efficiency loss) when scaling >1–2.5×")
print("  historical monthly spend. Strongest effect expected on retarget")
print("  (limited warm audience) and lookalike (audience saturation).")
print("- No change in organic, seasonality, or competitive pressure.")
print("- Edge effects of the 1-day lag ignored for monthly forecast.")
print("- Anomaly days already excluded from rate calculation.")

print("\nCampaign I would kill: sc_feed_video")
print("Evidence: lowest first-month ROAS (0.34), high CAC (₹832), low annual")
print("mix (9%). It is the least efficient use of the next rupee.")
print("Evidence that would prove me wrong: if a 4-week holdout showed")
print("incremental LTV (beyond first month) high enough to bring payback")
print("<30 days, or if ShareChat delivered unique users with markedly higher")
print("retention that Meta/Google do not reach.")

print("\nHeadline number: blended first-month net ROAS of 0.56")
print("Why: it is the single figure that tells the founder whether the ₹1 Cr")
print("is expected to return more or less than half its value in the first")
print("billing cycle — the practical cash constraint for a paid-growth app.")

# ---------------------------------------------------------------------------
# 5. Chart
# ---------------------------------------------------------------------------
print("\n=== TASK 5: CHART ===")
fig, ax = plt.subplots(figsize=(9, 5))
ordered = m['first_month_roas'].sort_values(ascending=True)
colors = ['#2ecc71' if r >= 0.8 else '#f39c12' if r >= 0.5 else '#e74c3c' for r in ordered]
bars = ax.barh(ordered.index, ordered.values, color=colors, height=0.6)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='Break-even ROAS')
ax.axvline(0.56, color='#3498db', linestyle=':', linewidth=1.5, label='Proposed blended 0.56')
for bar, val in zip(bars, ordered.values):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.2f}',
            va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('First-month net ROAS (net revenue ÷ spend)')
ax.set_title('Rasoi — First-month net ROAS by campaign\n(Reallocate toward green; cut red)', fontsize=13)
ax.legend(loc='lower right')
ax.set_xlim(0, 1.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
chart_path = OUT_DIR / 'roas_by_campaign.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
print(f"Chart saved: {chart_path}")
print("Sentence: Shift budget from red (ShareChat, Lookalike, UAC-Installs) to")
print("green (Retarget) and amber (Actions, Broad) until retarget saturates.")

# ---------------------------------------------------------------------------
# Save summary tables
# ---------------------------------------------------------------------------
m.round(3).to_csv(OUT_DIR / 'campaign_metrics.csv')
plan.to_csv(OUT_DIR / 'budget_plan.csv', index=False)
print("\nTables written to", OUT_DIR)
print("Done.")

=== TASK 1: DATA PROFILE ===
Rows: 540, Date range: 2026-06-01 → 2026-08-29
Campaigns: ['gg_uac_actions', 'gg_uac_installs', 'mt_broad_aoa', 'mt_lookalike_3pct', 'mt_retarget_paywall', 'sc_feed_video']
Channels: ['google', 'meta', 'sharechat']
No missing values. Revenue exactly matches trials*1 + monthly*199 + annual*1199.

*** ANOMALY FOUND: 12 rows ***
Google campaigns (gg_uac_installs + gg_uac_actions) 19–24 Jul 2026:
  - Spend, impressions, clicks, installs NORMAL
  - Trials, renewals, revenue collapsed to ~1–2% of normal levels
  - Trial CVR fell from ~6% to ~0.3–0.6%
Likely cause: purchase-event tracking / attribution outage (or payment
gateway issue) specific to Google traffic for those 6 days. Installs
continued to be recorded (different MMP path), but in-app trial purchases
were not attributed.
Rows touched: all Google rows for dates 2026-07-19 through 2026-07-24.
Handling: EXCLUDE these 12 rows from all conversion / revenue / ratio
calculations (CPT, CAC, trial-to-paid, ROAS)